# Training: Set Transformer + Deep Sets + Ensemble

Обучение моделей для предсказания результатов DOT

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'data/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load data
X_train = np.load(f'{DATA_PATH}/X_train.npy')
X_test = np.load(f'{DATA_PATH}/X_test.npy')
y_visc = np.load(f'{DATA_PATH}/y_visc.npy')
y_oxid = np.load(f'{DATA_PATH}/y_oxid.npy')
test_ids = pd.read_csv(f'{DATA_PATH}/test_ids.csv')['scenario_id'].tolist()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## Deep Sets Model

In [ ]:
class DeepSetsEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, output_dim=64):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.rho = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        phi_out = self.phi(x)
        rho_out = self.rho(phi_out.mean(dim=1, keepdim=True))
        return rho_out.squeeze()

class PredictorMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, output_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

class DeepSetsModel(nn.Module):
    def __init__(self, feature_dim, hidden_dim=128, encode_dim=64):
        super().__init__()
        self.encoder = DeepSetsEncoder(feature_dim, hidden_dim, encode_dim)
        self.predictor = PredictorMLP(encode_dim, hidden_dim, 1)
    
    def forward(self, x):
        encoded = self.encoder(x)
        return self.predictor(encoded)

print("Deep Sets defined")

## Set Transformer (ISAB)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class ISAB(nn.Module):
    def __init__(self, dim, num_heads=4, m=8):
        super().__init__()
        self.m = m
        self.I = nn.Parameter(torch.randn(1, m, dim))
        self.attn1 = MultiHeadAttention(dim, num_heads)
        self.attn2 = MultiHeadAttention(dim, num_heads)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
    
    def forward(self, x):
        B, N, C = x.shape
        I = self.I.repeat(B, 1, 1)
        h = self.norm1(self.attn1(torch.cat([I, x], dim=1))[:, :self.m])
        out = self.norm2(self.attn2(torch.cat([h, x], dim=1))[:, self.m:])
        return out

class SetTransformer(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_heads=4, num_isab=2, output_dim=1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, hidden_dim)
        self.isabs = nn.ModuleList([ISAB(hidden_dim, num_heads) for _ in range(num_isab)])
        self.norm = nn.LayerNorm(hidden_dim)
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        x = self.embedding(x)
        for isab in self.isabs:
            x = isab(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.predictor(x)

print("Set Transformer defined")

## Training Functions

In [ ]:
def train_model(model, X_train, y_train, X_test, epochs=100, lr=0.001, batch_size=32):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32)
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb).squeeze()
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
    
    return model

def predict_model(model, X):
    model.eval()
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    with torch.no_grad():
        pred = model(X_tensor).squeeze().cpu().numpy()
    return pred

def nmse(y_true, y_pred):
    y_range = y_true.max() - y_true.min()
    if y_range == 0:
        return 0
    return np.mean((y_true - y_pred) ** 2) / y_range

def nmae(y_true, y_pred):
    y_range = y_true.max() - y_true.min()
    if y_range == 0:
        return 0
    return np.mean(np.abs(y_true - y_pred)) / y_range

print("Functions defined")

## Train Models for Viscosity

In [ ]:
INPUT_DIM = X_train.shape[1]

# Deep Sets for Viscosity
print("Training Deep Sets - Viscosity...")
ds_visc = DeepSetsModel(INPUT_DIM, hidden_dim=128, encode_dim=64)
ds_visc = train_model(ds_visc, X_train, y_visc, X_test, epochs=150)

In [ ]:
# Set Transformer for Viscosity
print("Training Set Transformer - Viscosity...")
st_visc = SetTransformer(INPUT_DIM, hidden_dim=64, num_heads=4, num_isab=2)
st_visc = train_model(st_visc, X_train, y_visc, X_test, epochs=150)

## Train Models for Oxidation

In [ ]:
# Deep Sets for Oxidation
print("Training Deep Sets - Oxidation...")
ds_oxid = DeepSetsModel(INPUT_DIM, hidden_dim=128, encode_dim=64)
ds_oxid = train_model(ds_oxid, X_train, y_oxid, X_test, epochs=150)

In [ ]:
# Set Transformer for Oxidation
print("Training Set Transformer - Oxidation...")
st_oxid = SetTransformer(INPUT_DIM, hidden_dim=64, num_heads=4, num_isab=2)
st_oxid = train_model(st_oxid, X_train, y_oxid, X_test, epochs=150)

## Evaluate on Train Data

In [ ]:
# Predictions on train
pred_ds_visc = predict_model(ds_visc, X_train)
pred_st_visc = predict_model(st_visc, X_train)
pred_ds_oxid = predict_model(ds_oxid, X_train)
pred_st_oxid = predict_model(st_oxid, X_train)

# Ensemble
pred_ens_visc = (pred_ds_visc + pred_st_visc) / 2
pred_ens_oxid = (pred_ds_oxid + pred_st_oxid) / 2

print("=== Train Results ===")
print(f"Deep Sets Viscosity - NMAE: {nmae(y_visc, pred_ds_visc):.4f}, NMSE: {nmse(y_visc, pred_ds_visc):.4f}")
print(f"Set Trans Viscosity - NMAE: {nmae(y_visc, pred_st_visc):.4f}, NMSE: {nmse(y_visc, pred_st_visc):.4f}")
print(f"Ensemble Viscosity - NMAE: {nmae(y_visc, pred_ens_visc):.4f}, NMSE: {nmse(y_visc, pred_ens_visc):.4f}")
print()
print(f"Deep Sets Oxidation - NMAE: {nmae(y_oxid, pred_ds_oxid):.4f}, NMSE: {nmse(y_oxid, pred_ds_oxid):.4f}")
print(f"Set Trans Oxidation - NMAE: {nmae(y_oxid, pred_st_oxid):.4f}, NMSE: {nmse(y_oxid, pred_st_oxid):.4f}")
print(f"Ensemble Oxidation - NMAE: {nmae(y_oxid, pred_ens_oxid):.4f}, NMSE: {nmse(y_oxid, pred_ens_oxid):.4f}")

## Predict on Test

In [ ]:
# Predictions on test
test_ds_visc = predict_model(ds_visc, X_test)
test_st_visc = predict_model(st_visc, X_test)
test_ds_oxid = predict_model(ds_oxid, X_test)
test_st_oxid = predict_model(st_oxid, X_test)

# Ensemble
test_pred_visc = (test_ds_visc + test_st_visc) / 2
test_pred_oxid = (test_ds_oxid + test_st_oxid) / 2

print(f"Test Viscosity: mean={test_pred_visc.mean():.2f}, std={test_pred_visc.std():.2f}")
print(f"Test Oxidation: mean={test_pred_oxid.mean():.2f}, std={test_pred_oxid.std():.2f}")

## Save Submission

In [ ]:
import os
os.makedirs('submissions', exist_ok=True)
os.makedirs('models', exist_ok=True)

TARGET_VISC = 'Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %'
TARGET_OXID = 'Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm'

submission = pd.DataFrame({
    'scenario_id': test_ids,
    TARGET_VISC: test_pred_visc,
    TARGET_OXID: test_pred_oxid
})

submission.to_csv('submissions/predictions.csv', index=False)
print(f"Submission saved: {submission.shape}")
print(submission.head())

In [ ]:
# Save models
torch.save(ds_visc.state_dict(), 'models/ds_visc.pth')
torch.save(ds_oxid.state_dict(), 'models/ds_oxid.pth')
torch.save(st_visc.state_dict(), 'models/st_visc.pth')
torch.save(st_oxid.state_dict(), 'models/st_oxid.pth')
print("Models saved to models/")